# 05주 · 직렬·병렬·분기관의 연속식

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 직렬·병렬 관로 / 다지관수로

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week05/CE272_W05.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week05) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 주제별 실습 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 고정 f 병렬관. 가변 f 해석은 Colab


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

<a id="s01"></a>

## 같은 수두차를 갖는 병렬관

**먼저 예측**: 직경이 두 배인 관의 유량도 두 배인가?

**문제**: 수두차 20 m인 수조 사이에 길이 500 m, 직경 0.2·0.3·0.4 m인 관 3개가 병렬 연결되어 있다. 모든 Darcy 마찰계수를 0.02로 고정하고 각 관의 유량과 총유량을 구하시오. 국부손실은 무시하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
L = np.array([500.,500.,500.])
D = np.array([.2,.3,.4])
f = np.array([.02,.02,.02])
H = 20.
Q = hyd.parallel_fixed(H,L,D,f)
head_checks = [hyd.pipe_loss(q,l,d,fixed_f=v) for q,l,d,v in zip(Q,L,D,f)]
display(pd.DataFrame({'D_m':D,'Q_m3_s':Q,'headloss_m':head_checks,'Q_per_D_2_5':Q/D**2.5}))
print('Total Q:',Q.sum(),'m³/s')

**검토 문제**: 같은 수두차와 고정 f에서 Q∝D^(5/2)/√L 관계를 유도하시오. Q∝A/L을 그대로 적용하면 왜 맞지 않는지 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


<a id="s02"></a>

## 마찰계수를 다시 계산하는 병렬관

**먼저 예측**: 총유량만 알 때 관별 유량을 단면적 비율로 바로 나눌 수 있는가?

**문제**: 총유량 0.6 m³/s를 길이 500·400·600 m, 직경 0.3·0.2·0.4 m인 병렬관에 분배하시오. 모든 관의 조도는 0.2 mm, ν=1.006×10⁻⁶ m²/s이다. 공통 손실수두와 연속식 잔차를 확인하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
lengths, diameters = [500.,400.,600.],[.3,.2,.4]
segments = [[dict(L=L,D=D,epsilon=.0002)] for L,D in zip(lengths,diameters)]
nu, Q_total = 1.006e-6,.6
# 모든 분기관이 완전 난류인 공통 수두차 구간을 먼저 확보
def flows(H):
    return np.array([hyd.turbulent_flow(H,s,nu) for s in segments])
head = brentq(lambda H:flows(H).sum()-Q_total,.1,100.)
Q = flows(head)
display(pd.DataFrame({'L_m':lengths,'D_m':diameters,'Q_m3_s':Q,
                      'headloss_m':[hyd.pipe_loss(q,L,D,nu,.0002) for q,L,D in zip(Q,lengths,diameters)]}))
print('Common head:',head,'m; mass residual:',Q.sum()-Q_total,'m³/s')

**검토 문제**: 분기점 수두를 미지수로 두는 방법을 세 저수지 문제와 연결하시오. 흐름 방향이 처음 가정과 다르면 음의 유량을 어떻게 해석할지 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


## 학습 기록

절점 연속식과 경로별 에너지 조건을 동시에 만족하는지 검토하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록


In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')